# 🔍 Inference Analysis

Análisis completo de la salida de inferencia del framework Energizados.

**Qué hace esta notebook:**
- Carga el CSV de predicciones y su metadata
- Distribución de probabilidades (global y por región)
- Análisis por umbrales (globales y segmentados)
- Estratificación de riesgo por deciles
- Top clientes más sospechosos
- Patrones de consumo vs probabilidad de fraude
- Export de resultados

## 0. Configuración

Acá definimos las rutas a los archivos de entrada. **Son las únicas variables que necesitás cambiar** para analizar otra corrida de inferencia.

In [ ]:
# ============================================================
# CONFIGURACIÓN — paths, archivos y parámetros
# ============================================================
# Toda la configuración está acá. Para apuntar la notebook a otro
# proyecto o dataset, modificá estos valores (no hace falta tocar el
# resto de las celdas).

from pathlib import Path

# --- Proyecto y datos ---
PROJECT_PATH = Path('/home/vvv/Develop/bid/energizados/.proyects/sample')
OUTPUT_PATH = PROJECT_PATH / 'output'
VERSION = 'v5'
INFERENCE_DIR_NAME = 'inference-20260808_1254'
INFERENCE_DIR = OUTPUT_PATH / VERSION / INFERENCE_DIR_NAME

# --- Archivos de inferencia ---
PREDICTIONS_CSV = INFERENCE_DIR / 'predictions.csv'
METADATA_JSON = INFERENCE_DIR / 'predictions.csv.metadata.json'

# --- Umbrales por segmento (geo_region) ---
SEGMENT_THRESHOLDS_PATH = OUTPUT_PATH / VERSION / 'segment_thresholds_geo_region.json'

# --- Entorno (detección automática Colab vs local) ---
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'PROJECT_PATH           : {PROJECT_PATH}')
print(f'INFERENCE_DIR          : {INFERENCE_DIR}')
print(f'PREDICTIONS_CSV        : {PREDICTIONS_CSV}')
print(f'SEGMENT_THRESHOLDS_PATH: {SEGMENT_THRESHOLDS_PATH}')
print(f'IN_COLAB               : {IN_COLAB}')

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Estilo
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (12, 6)

# Verificar que los archivos existen
for p in [PREDICTIONS_CSV, METADATA_JSON]:
    assert Path(p).exists(), f"No se encuentra: {p}"

print("✅ Configuración lista")

## 1. Carga de datos

Cargamos el CSV de predicciones y el archivo de metadata que lo acompaña.

**Qué mirar:**
- `row_count` en la metadata debe coincidir con la cantidad de filas del CSV.
- `threshold` es el umbral por defecto que usó la inferencia para decidir fraude/no-fraude.
- `model_hash` identifica el modelo exacto que generó estas predicciones (trazabilidad).
- `output_columns` confirma qué columnas incluye el CSV.
- La validación final chequea integridad: misma cantidad de filas, probabilidades en [0,1], y `cliente` único (sin duplicados).

In [ ]:
# Cargar metadata
with open(METADATA_JSON) as f:
    metadata = json.load(f)

print("=== Metadata ===")
for k, v in metadata.items():
    print(f"  {k}: {v}")

In [ ]:
# Cargar predicciones
df = pd.read_csv(PREDICTIONS_CSV)

# Columnas de consumo
CONSUMPTION_COLS = [c for c in df.columns if c.endswith("_anterior")]

print(f"Shape: {df.shape}")
print(f"Columnas: {list(df.columns)}")
print(f"Memoria: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
df.head(3)

In [ ]:
# Validación básica
assert len(df) == metadata["row_count"], "Row count mismatch con metadata"
assert df["probability"].between(0, 1).all(), "Probabilidades fuera de [0,1]"
assert df["cliente"].is_unique, "cliente no es único"
print("✅ Validación OK")

## 2. Distribución de probabilidades

Esta sección muestra cómo se distribuyen las probabilidades de fraude en toda la población evaluada.

**Qué mirar en el histograma:**
- **Forma general:** ¿Hay dos montañas claras (clientes "normales" vs "sospechosos") o es una distribución más uniforme? Un modelo con buena separación muestra concentración en los extremos.
- **Densidad cerca de 0.5:** Muchos clientes con probabilidad ~0.5 indican que el modelo no logra decidir — son casos "grises" donde el modelo tiene baja confianza.
- **Masa en el extremo derecho (prob > 0.8):** Son los clientes donde el modelo está más seguro del fraude. Si hay demasiados, puede indicar sobreajuste o features que "filtran" el target.
- **Masa en el extremo izquierdo (prob < 0.2):** Clientes que el modelo considera casi seguro que NO son fraude.

**Qué mirar en el boxplot:**
- La mediana (línea naranja) te dice el "centro" de las predicciones.
- Los bigotes y outliers muestran dispersión. Un boxplot muy compacto arriba de 0.8 indica que el modelo está muy sesgado a predecir alto.

**Qué mirar en la ECDF (función de distribución acumulada empírica):**
- Es la forma más precisa de ver cuántos clientes están por debajo de cada probabilidad.
- Trazá una línea horizontal en 0.5 y leé el valor en X: te dice la mediana.
- Trazá una línea vertical en 0.5 y leé en Y: te dice qué proporción tiene probabilidad ≤ 0.5.
- Una ECDF "escalonada" fuerte sugiere que el modelo colapsa muchas predicciones al mismo valor (posible saturación).

**Estadísticas:**
- `skewness` negativa = masa cargada a la derecha (muchos predichos como fraude). Positiva = cargada a la izquierda (pocos fraude).
- `kurtosis` alta = distribución muy picuda (mucha concentración en pocos valores).

In [ ]:
prob = df["probability"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histograma
axes[0].hist(prob, bins=80, color="steelblue", edgecolor="white", alpha=0.85)
axes[0].axvline(0.5, color="crimson", linestyle="--", linewidth=1.5, label="threshold=0.5")
axes[0].set_title("Distribución de probabilidades")
axes[0].set_xlabel("Probability")
axes[0].set_ylabel("Frecuencia")
axes[0].legend()

# Boxplot
axes[1].boxplot(prob, vert=True, patch_artist=True,
                boxprops=dict(facecolor="steelblue", alpha=0.6))
axes[1].set_title("Boxplot")
axes[1].set_ylabel("Probability")

# ECDF
sorted_prob = np.sort(prob)
y_ecdf = np.arange(1, len(sorted_prob) + 1) / len(sorted_prob)
axes[2].plot(sorted_prob, y_ecdf, color="steelblue", linewidth=1.5)
axes[2].axvline(0.5, color="crimson", linestyle="--", linewidth=1.5)
axes[2].set_title("ECDF")
axes[2].set_xlabel("Probability")
axes[2].set_ylabel("Proporción acumulada")

plt.tight_layout()
plt.show()

In [ ]:
# Estadísticas detalladas
stats = prob.describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
print("=== Estadísticas de probabilidad ===")
print(stats.to_string())
print(f"\nSkewness: {prob.skew():.4f}")
print(f"Kurtosis: {prob.kurtosis():.4f}")

## 3. Análisis por umbrales

Acá vemos cuántos clientes quedarían marcados como fraude a distintos niveles de threshold.

**Qué mirar en la curva de threshold:**
- Es una herramienta de **decisión operativa**: según cuántos casos tu equipo puede investigar, elegís el threshold.
- La curva de % muestra cuánto del total de clientes se marcaría. Ej: si threshold=0.7 y se marca el 60% de los clientes, probablemente el modelo tiene poca discriminación.
- **Codo en la curva:** ¿Hay un punto donde la caída es muy abrupta? Ese threshold puede ser un buen candidato si querés maximizar precisión sin perder demasiado recall.
- Compará con el threshold por defecto de la metadata: ¿es razonable para tu caso de uso?

**Regla de negocio típica:**
- Umbral bajo (0.3-0.4) → muchos marcados, priorizás recall (no se te escapa ningún fraude pero inspeccionás muchos falsos positivos).
- Umbral alto (0.7-0.8) → pocos marcados, priorizás precisión (los que marcás casi seguro son fraude pero dejás pasar varios).

In [ ]:
# Curva de threshold: cuántos clientes quedan marcados a cada nivel
thresholds = np.arange(0.1, 1.0, 0.05)
counts = [(prob >= t).sum() for t in thresholds]
pcts = [c / len(prob) * 100 for c in counts]

fig, ax1 = plt.subplots(figsize=(12, 5))

color = "steelblue"
ax1.bar(range(len(thresholds)), counts, color=color, alpha=0.7, width=0.6)
ax1.set_xlabel("Threshold")
ax1.set_ylabel("Clientes marcados (≥ threshold)", color=color)
ax1.set_xticks(range(len(thresholds)))
ax1.set_xticklabels([f"{t:.2f}" for t in thresholds], rotation=45)
ax1.tick_params(axis="y", labelcolor=color)

ax2 = ax1.twinx()
ax2.plot(range(len(thresholds)), pcts, "o-", color="crimson", linewidth=2, markersize=6)
ax2.set_ylabel("% del total", color="crimson")
ax2.tick_params(axis="y", labelcolor="crimson")

plt.title("Clientes marcados por threshold")
plt.tight_layout()
plt.show()

# Tabla
threshold_df = pd.DataFrame({
    "threshold": thresholds,
    "marcados": counts,
    "%_del_total": [f"{p:.2f}%" for p in pcts],
})
threshold_df

In [ ]:
# Threshold por defecto de la metadata
default_threshold = metadata["threshold"]
marked = (prob >= default_threshold).sum()
print(f"Threshold por defecto: {default_threshold}")
print(f"Clientes marcados: {marked:,} ({marked / len(prob) * 100:.2f}%)")
print(f"Clientes NO marcados: {len(prob) - marked:,} ({(len(prob) - marked) / len(prob) * 100:.2f}%)")

## 4. Análisis por región y segment thresholds

Cuando la inferencia usa **segment thresholds**, cada región (u otro segmento) tiene su propio umbral de decisión, calibrado según las características de esa población. Esto es más justo que un threshold único porque regiones con distintas tasas de fraude necesitan distintos puntos de corte.

**Qué mirar en los histogramas por región:**
- ¿Las distribuciones tienen formas muy distintas entre regiones? Si una región concentra probabilidades altas y otra bajas, puede haber un **sesgo regional** en el modelo (overfitting a patrones de una región).
- Compará cuántos clientes caen a la derecha del threshold en cada región.

**Qué mirar en los boxplots:**
- La posición de las cajas te dice si el modelo "favorece" ciertas regiones con scores más altos.
- Si una región tiene mediana mucho más alta que otra, investigá si tiene sentido desde el negocio (¿realmente hay más fraude ahí?) o es un artefacto del entrenamiento.

**Qué mirar en la tabla de segment thresholds:**
- `positive_rate`: tasa real de fraude en los datos de validación para ese segmento. Segmentos con `positive_rate` muy bajo suelen tener thresholds altos o infinitos (el modelo no logra diferenciar).
- `auc`: capacidad discriminativa. AUC < 0.6 es malo — el modelo no separa bien en ese segmento. AUC cercano a 0.5 es aleatorio.
- `threshold`: si es `Infinity`, ese segmento no tiene un punto de corte útil (muy pocos o ningún positivo en validación). Los clientes de esos segmentos se marcan solo si superan el threshold global.
- `f1`: balance precisión-recall. Un F1 muy bajo (< 0.2) indica que el modelo no funciona bien en esa región.

**Qué mirar en la comparación global vs segmentado:**
- El Δ (delta) muestra cuántos clientes **adicionales** se marcarían o dejarían de marcarse al usar el threshold del segmento en vez del global.
- Un Δ muy positivo (muchos más marcados) puede indicar que el threshold del segmento es muy laxo; uno muy negativo, que es muy estricto.

In [ ]:
# Distribución por región
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, region in zip(axes, df["geo_region"].unique()):
    region_prob = df[df["geo_region"] == region]["probability"]
    ax.hist(region_prob, bins=60, color="steelblue", edgecolor="white", alpha=0.8)
    ax.axvline(0.5, color="crimson", linestyle="--", linewidth=1.5, label="threshold=0.5")
    ax.set_title(f"{region} (n={len(region_prob):,})")
    ax.set_xlabel("Probability")
    ax.set_ylabel("Frecuencia")
    ax.legend()

plt.suptitle("Distribución de probabilidad por región", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots comparativos por región
fig, ax = plt.subplots(figsize=(10, 5))
df.boxplot(column="probability", by="geo_region", ax=ax, patch_artist=True,
           boxprops=dict(facecolor="steelblue", alpha=0.6))
ax.set_title("Probabilidad por región")
ax.set_xlabel("Región")
ax.set_ylabel("Probability")
plt.suptitle("")
plt.tight_layout()
plt.show()

# Stats por región
region_stats = df.groupby("geo_region")["probability"].describe()
region_stats[">= 0.5"] = df.groupby("geo_region")["probability"].apply(lambda x: (x >= 0.5).sum())
region_stats[">= 0.5 %"] = df.groupby("geo_region")["probability"].apply(lambda x: (x >= 0.5).mean() * 100)
region_stats

In [ ]:
# Cargar y visualizar segment thresholds
seg_path = Path(SEGMENT_THRESHOLDS_PATH)
if seg_path.exists():
    with open(seg_path) as f:
        seg_thresholds = json.load(f)
    
    print(f"Segment column: {seg_thresholds['segment_column']}")
    print(f"Threshold mode: {seg_thresholds['threshold_mode']}")
    print(f"Default threshold: {seg_thresholds['default_threshold']}")
    print(f"Número de segmentos: {len(seg_thresholds['segments'])}")
    print()
    
    # Tabla de thresholds
    seg_rows = []
    for seg_name, seg_data in seg_thresholds["segments"].items():
        seg_rows.append({
            "segmento": seg_name,
            "n_samples": seg_data["n_samples"],
            "n_positives": seg_data["n_positives"],
            "positive_rate": f"{seg_data['positive_rate']:.4f}",
            "auc": f"{seg_data['auc']:.4f}",
            "f1": f"{seg_data['f1']:.4f}",
            "threshold": f"{seg_data['threshold']:.4f}",
        })
    seg_df = pd.DataFrame(seg_rows)
    display(seg_df)
else:
    print("⚠️ No se encontró archivo de segment thresholds")

In [ ]:
# Comparar thresholds global vs segmentados para las regiones en los datos
if seg_path.exists():
    fig, axes = plt.subplots(1, len(df["geo_region"].unique()), figsize=(14, 5))
    if len(df["geo_region"].unique()) == 1:
        axes = [axes]
    
    for ax, region in zip(axes, sorted(df["geo_region"].unique())):
        region_prob = df[df["geo_region"] == region]["probability"]
        ax.hist(region_prob, bins=60, color="steelblue", edgecolor="white", alpha=0.7)
        
        # Threshold global
        ax.axvline(default_threshold, color="gray", linestyle="--", linewidth=1.5, label=f"global={default_threshold}")
        
        # Threshold del segmento
        seg_info = seg_thresholds["segments"].get(region)
        if seg_info and seg_info["threshold"] != float("inf"):
            seg_t = seg_info["threshold"]
            ax.axvline(seg_t, color="crimson", linestyle="-", linewidth=2, label=f"segment={seg_t:.4f}")
            
            # Cuántos cambia
            global_marked = (region_prob >= default_threshold).sum()
            seg_marked = (region_prob >= seg_t).sum()
            delta = seg_marked - global_marked
            ax.set_title(f"{region}\n(global: {global_marked:,} | segment: {seg_marked:,} | Δ={delta:+,})")
        else:
            ax.set_title(f"{region} (sin segment threshold)")
        
        ax.set_xlabel("Probability")
        ax.legend(fontsize=8)
    
    plt.suptitle("Threshold global vs segmentado", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

## 5. Estratificación de riesgo por deciles

Dividimos a los clientes en 10 grupos de igual tamaño ordenados por probabilidad de fraude. El **decil 1** contiene al 10% más sospechoso; el **decil 10** al 10% menos sospechoso.

**Qué mirar:**
- **Rango de probabilidad por decil:** ¿Los deciles están bien separados? Si el decil 1 y el decil 5 tienen rangos que se solapan mucho, el modelo no está rankeando bien.
- **Deciles 1-3:** Son tus clientes de mayor prioridad. Si el modelo funciona bien, acá debería estar la mayoría de los fraudes reales.
- **Consumo medio por decil:** ¿Hay una tendencia clara? En fraude eléctrico es común ver que los deciles de alto riesgo tienen **consumo más bajo** (porque el fraude reduce el registro de consumo). Si no ves esa relación, el modelo puede estar apoyándose en otros patrones (o no estar capturando esta señal clásica).
- **Decil con consumo cero o muy bajo + probabilidad alta:** Es un patrón clásico de fraude (medidor puenteado o manipulado). Si el decil 1 tiene consumo medio bajo y probabilidad alta, es buena señal.
- Si todos los deciles tienen probabilidad > 0.5, tenés un problema de calibración: el modelo está sobreestimando el riesgo para casi todos.

In [ ]:
# Asignar deciles (1 = más sospechoso, 10 = menos sospechoso)
df["decile"] = pd.qcut(df["probability"], 10, labels=range(10, 0, -1), duplicates="drop")
df["decile"] = df["decile"].astype(int)

# Resumen por decil
decile_summary = df.groupby("decile").agg(
    n=("cliente", "count"),
    prob_min=("probability", "min"),
    prob_mean=("probability", "mean"),
    prob_max=("probability", "max"),
    consumo_medio=("1_anterior", "mean"),
).sort_index()

decile_summary["%_acum"] = decile_summary["n"].cumsum() / len(df) * 100
decile_summary

In [ ]:
# Visualización de deciles
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Barras: clientes por decil
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.9, 10))
axes[0].bar(decile_summary.index, decile_summary["n"], color=colors, edgecolor="white")
axes[0].set_title("Clientes por decil")
axes[0].set_xlabel("Decil (1 = más riesgo)")
axes[0].set_ylabel("Clientes")
axes[0].set_xticks(range(1, 11))

# Rango de probabilidad por decil
for i, row in decile_summary.iterrows():
    axes[1].bar(i, row["prob_max"] - row["prob_min"], bottom=row["prob_min"],
               color=colors[i - 1], edgecolor="white")
axes[1].axhline(0.5, color="gray", linestyle="--", linewidth=1, label="threshold=0.5")
axes[1].set_title("Rango de probabilidad por decil")
axes[1].set_xlabel("Decil")
axes[1].set_ylabel("Probability")
axes[1].legend()

# Consumo medio por decil
axes[2].bar(decile_summary.index, decile_summary["consumo_medio"], color=colors, edgecolor="white")
axes[2].set_title("Consumo medio (1_anterior) por decil")
axes[2].set_xlabel("Decil (1 = más riesgo)")
axes[2].set_ylabel("Consumo medio (kWh)")

plt.tight_layout()
plt.show()

## 6. Top clientes más sospechosos

Lista de los clientes con mayor probabilidad de fraude, ordenados de mayor a menor.

**Qué mirar:**
- **Concentración geográfica:** ¿Los top N son todos de la misma región? Si es así, puede ser un sesgo o una región con fraude estructural.
- **Rango de probabilidad:** Si el top 50 tiene probabilidades muy similares entre sí (ej. 0.9361 a 0.9360), el modelo está saturando en el extremo — no discrimina entre los más sospechosos.
- **Perfil de consumo:** ¿Los top sospechosos tienen consumo 0 en el último mes? ¿Caídas abruptas en los últimos 3 meses? ¿Consumo constante (flat) todo el año? Estos son patrones operativos clásicos.
- **Cantidad de clientes en banda de alto riesgo (≥ 0.9):** Si es un número muy alto (ej. > 50% de la población), el modelo está sobrecalibrado hacia arriba. Revisá si el threshold de entrenamiento era correcto.

**Tip operativo:** La tabla de top N podés exportarla (sección 10) y compartirla con el equipo de inspección para que prioricen.

In [ ]:
TOP_N = 50
top_suspicious = df.nlargest(TOP_N, "probability")[
    ["cliente", "geo_region", "probability"] + CONSUMPTION_COLS
]

print(f"=== Top {TOP_N} clientes más sospechosos ===")
print(f"Rango de probabilidad: [{top_suspicious['probability'].min():.6f}, {top_suspicious['probability'].max():.6f}]")
print(f"Regiones: {top_suspicious['geo_region'].value_counts().to_dict()}")
top_suspicious

In [ ]:
# Top N con probabilidad > 0.9 agrupados por región (más útil para inspección)
high_risk = df[df["probability"] >= 0.9].copy()
print(f"Clientes con prob >= 0.9: {len(high_risk):,} ({len(high_risk) / len(df) * 100:.2f}%)")
print(f"Por región:")
for region, group in high_risk.groupby("geo_region"):
    print(f"  {region}: {len(group):,}")

## 7. Patrones de consumo vs probabilidad

Esta sección es la más importante desde el punto de vista operativo: **¿el modelo está capturando los patrones de consumo que esperamos ver en fraude?**

**Qué mirar en los scatter plots:**
- Se usa una muestra de 20k puntos por performance. Los puntos son clientes individuales.
- **Consumo bajo + probabilidad alta (esquina superior izquierda):** Es lo que esperamos — poco consumo registrado, alta sospecha. Si hay muchos puntos acá, el modelo funciona intuitivamente.
- **Consumo alto + probabilidad alta (esquina superior derecha):** Clientes que consumen mucho pero igual son marcados. Pueden ser comercios/industrias legítimas con patrones atípicos, o fraude a gran escala.
- **Consumo bajo + probabilidad baja (esquina inferior izquierda):** Clientes de bajo consumo no marcados. Pueden ser viviendas pequeñas legítimas.
- **Nube difusa sin patrón claro:** El modelo no está usando el consumo como señal principal — puede estar apoyándose en features categóricas (región, tipo de tarifa, etc.).

**Qué mirar en los boxplots de consumo por bucket:**
- Compará la mediana de consumo entre buckets. Si el bucket "0.9+" tiene mediana de consumo mucho más baja que "<0.3", el modelo está capturando la señal de "menos consumo = más fraude".
- Si las cajas son muy similares entre buckets, el consumo no está siendo discriminativo.

**Qué mirar en los perfiles mensuales (curvas):**
- La línea es la **mediana** de consumo para cada mes hacia atrás (12 = hace 12 meses, 1 = último mes).
- La banda sombreada es el rango intercuartil (percentiles 25-75).
- **Caída abrupta en los últimos meses en buckets de alta probabilidad:** Patrón clásico de fraude — el cliente manipuló el medidor recientemente.
- **Línea plana todo el año en buckets de alta probabilidad:** Consumo sospechosamente constante, típico de medidores puenteados que registran un valor fijo.
- **Línea con estacionalidad normal (picos en invierno/verano) en buckets de baja probabilidad:** Comportamiento esperado de un cliente legítimo.
- Compará la **forma** de las curvas entre buckets, no solo la altura. Dos curvas paralelas separadas solo por nivel pueden indicar que el modelo solo aprendió "poco consumo = fraude" sin capturar patrones de cambio.

In [ ]:
# Scatter: consumo último mes vs probabilidad (samplear para performance)
sample_n = min(20_000, len(df))
df_sample = df.sample(n=sample_n, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Consumo último mes
sc = axes[0].scatter(
    df_sample["1_anterior"], df_sample["probability"],
    c=df_sample["probability"], cmap="RdYlGn_r", alpha=0.3, s=2
)
axes[0].set_xlabel("Consumo 1_anterior (kWh)")
axes[0].set_ylabel("Probability")
axes[0].set_title("Consumo último mes vs Probabilidad")
axes[0].set_xlim(0, df_sample["1_anterior"].quantile(0.99))
plt.colorbar(sc, ax=axes[0], label="Probability")

# Consumo medio 12 meses
df_sample["consumo_medio_12m"] = df_sample[CONSUMPTION_COLS].mean(axis=1)
sc2 = axes[1].scatter(
    df_sample["consumo_medio_12m"], df_sample["probability"],
    c=df_sample["probability"], cmap="RdYlGn_r", alpha=0.3, s=2
)
axes[1].set_xlabel("Consumo medio 12 meses (kWh)")
axes[1].set_ylabel("Probability")
axes[1].set_title("Consumo medio 12 meses vs Probabilidad")
axes[1].set_xlim(0, df_sample["consumo_medio_12m"].quantile(0.99))
plt.colorbar(sc2, ax=axes[1], label="Probability")

plt.tight_layout()
plt.show()

In [ ]:
# Boxplots de consumo por bucket de probabilidad
df["prob_bucket"] = pd.cut(
    df["probability"],
    bins=[0, 0.3, 0.5, 0.7, 0.8, 0.9, 1.0],
    labels=["<0.3", "0.3-0.5", "0.5-0.7", "0.7-0.8", "0.8-0.9", "0.9+"]
)

# Figura 1: boxplots de consumo por bucket (3 meses clave)
fig1, axes1 = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes1, ["1_anterior", "6_anterior", "12_anterior"]):
    df.boxplot(column=col, by="prob_bucket", ax=ax, showfliers=False,
               patch_artist=True, boxprops=dict(facecolor="steelblue", alpha=0.6))
    ax.set_title(f"{col} por bucket de probabilidad")
    ax.set_xlabel("Probabilidad")
    ax.set_ylabel("Consumo (kWh)")
    plt.sca(ax)
    plt.xticks(rotation=45)
plt.suptitle("Distribución de consumo por bucket de probabilidad", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Figura 2: perfil de consumo mensual por bucket (mediana + IQR)
fig2, axes2 = plt.subplots(2, 3, figsize=(18, 10))
for i, bucket in enumerate(df["prob_bucket"].cat.categories):
    row, col = divmod(i, 3)
    ax = axes2[row, col]
    bucket_data = df[df["prob_bucket"] == bucket]
    median_curve = bucket_data[CONSUMPTION_COLS].median()
    months = list(range(12, 0, -1))
    ax.plot(months, median_curve.values, "o-", color="steelblue", linewidth=1.5, markersize=4)
    ax.fill_between(
        months,
        bucket_data[CONSUMPTION_COLS].quantile(0.25),
        bucket_data[CONSUMPTION_COLS].quantile(0.75),
        alpha=0.2, color="steelblue"
    )
    ax.set_title(f"Bucket {bucket} (n={len(bucket_data):,})")
    ax.set_xlabel("Meses atrás")
    ax.set_ylabel("Consumo mediano (kWh)")
    ax.invert_xaxis()
plt.suptitle("Patrones de consumo por bucket de probabilidad", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 8. Consumo cero y patrones extremos

El consumo cero es la señal más fuerte de fraude en distribución eléctrica: un medidor manipulado, puenteado o dado de baja irregularmente no registra consumo.

**Qué mirar:**
- **% de clientes con consumo cero en el último mes:** Si es muy alto, revisá que no sean bajas administrativas legítimas (medidores retirados, inmuebles deshabitados).
- **Probabilidad media de los consumo=0:** Si es alta (> 0.7), el modelo aprendió correctamente esta señal. Si es baja, el modelo no la está capturando — posiblemente porque en entrenamiento había pocos ejemplos de este patrón.
- **Consumo bajo (≤ P10) + probabilidad ≥ 0.9:** Estos son los casos más accionables: poco consumo y alta sospecha. Deberían ser prioridad 1 para inspección en campo.

**Histograma comparativo (consumo=0 vs consumo>0):**
- Si las dos distribuciones están nítidamente separadas (consumo=0 concentrado en probabilidades altas, consumo>0 en bajas), el modelo tiene una señal clara y saludable.
- Si se solapan mucho, el modelo está usando otras variables además del nivel de consumo — lo cual no es malo, pero entendé qué otras señales está usando.

**Probabilidad media por meses con consumo cero:**
- Esperamos ver una tendencia creciente: más meses en cero = más probabilidad de fraude.
- Un cliente con 12 meses en cero debería tener probabilidad muy alta. Si no es así, investigá.
- Un cliente con 0 meses en cero pero probabilidad alta está siendo marcado por otros patrones (ej. caídas abruptas, estacionalidad anómala).

In [ ]:
# Consumo cero en el último mes vs probabilidad
zero_consumption = df[df["1_anterior"] == 0]
print(f"Clientes con consumo cero en el último mes: {len(zero_consumption):,} ({len(zero_consumption) / len(df) * 100:.2f}%)")
print(f"  Probabilidad media: {zero_consumption['probability'].mean():.4f}")
print(f"  Prob >= 0.5: {(zero_consumption['probability'] >= 0.5).sum():,}")
print(f"  Prob >= 0.9: {(zero_consumption['probability'] >= 0.9).sum():,}")
print()

# Clientes con consumo muy bajo pero alta probabilidad (señal fuerte de fraude)
q10_consumo = df["1_anterior"].quantile(0.10)
low_cons_high_prob = df[(df["1_anterior"] <= q10_consumo) & (df["probability"] >= 0.9)]
print(f"Consumo <= P10 ({q10_consumo:.0f} kWh) Y prob >= 0.9: {len(low_cons_high_prob):,}")

In [ ]:
# Comparación: distribución de probabilidad para clientes con y sin consumo cero
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

has_zero_1m = df["1_anterior"] == 0

axes[0].hist(df[has_zero_1m]["probability"], bins=50, alpha=0.6, color="crimson", label="Consumo=0")
axes[0].hist(df[~has_zero_1m]["probability"], bins=50, alpha=0.6, color="steelblue", label="Consumo>0")
axes[0].set_title("Probabilidad: consumo cero vs positivo (último mes)")
axes[0].set_xlabel("Probability")
axes[0].legend()

# Cantidad de meses con consumo cero vs probabilidad
df["meses_cero"] = (df[CONSUMPTION_COLS] == 0).sum(axis=1)
zero_months_prob = df.groupby("meses_cero")["probability"].agg(["mean", "count"])
axes[1].bar(zero_months_prob.index, zero_months_prob["mean"], color="steelblue", edgecolor="white")
axes[1].set_title("Probabilidad media por cantidad de meses con consumo cero")
axes[1].set_xlabel("Meses con consumo = 0")
axes[1].set_ylabel("Probabilidad media")

plt.tight_layout()
plt.show()

zero_months_prob

## 8c. Perfil comparativo: decil 1 vs decil 10

¿Qué hace distintos a los clientes más sospechosos de los menos sospechosos? Comparamos sistemáticamente las features entre el decil 1 (top 10% más sospechoso) y el decil 10 (10% menos sospechoso).

**Qué mirar:**
- Features con mayor diferencia porcentual entre ambos extremos: son las que el modelo está usando para discriminar
- Si una feature que esperás que sea importante (ej. consumo cero) **no** aparece en el top, el modelo podría no estar capturándola correctamente
- Si una feature administrativa (ej. `tipo_tarifa_prob`) aparece como top driver, puede indicar un sesgo — el modelo está detectando correlaciones espurias en vez de fraude real

> Esta tabla es una de las más útiles para explicar el modelo a stakeholders no técnicos: "los clientes más sospechosos tienen en promedio un 80% menos consumo en el último mes y 3 veces más meses con consumo cero".

In [ ]:
# --- Perfil comparativo: decil 1 (top sospechosos) vs decil 10 (menos sospechosos) ---
decil1 = df[df['decile'] == 1]
decil10 = df[df['decile'] == 10]

print(f'Decil 1 (más sospechosos): {len(decil1):,} clientes')
print(f'Decil 10 (menos sospechosos): {len(decil10):,} clientes')

# Seleccionar features numéricas para comparar
feature_cols = [c for c in df.columns if c not in [
    'probability', 'cliente', 'confidence_zone', 'decile', 'prob_bucket',
    'geo_region', 'prediction'
]]
numeric_features = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
# Limitar a features más informativas (excluir IDs, hashes)
numeric_features = [c for c in numeric_features if not c.startswith('Unnamed')][:40]

# Construir tabla comparativa
profile = pd.DataFrame({
    'decil1_mean': decil1[numeric_features].mean(),
    'decil1_median': decil1[numeric_features].median(),
    'decil10_mean': decil10[numeric_features].mean(),
    'decil10_median': decil10[numeric_features].median(),
})
profile['diff_mean'] = profile['decil1_mean'] - profile['decil10_mean']
profile['pct_diff_mean'] = (profile['diff_mean'] / profile['decil10_mean'].abs().replace(0, np.nan)) * 100
profile = profile.sort_values('pct_diff_mean', key=abs, ascending=False).dropna(subset=['pct_diff_mean'])

print(f'\nTop 15 features que más diferencian al decil 1 del decil 10:')
display(profile.head(15).style
    .format({
        'decil1_mean': '{:.3f}', 'decil1_median': '{:.3f}',
        'decil10_mean': '{:.3f}', 'decil10_median': '{:.3f}',
        'diff_mean': '{:.3f}', 'pct_diff_mean': '{:.1f}%'
    })
    .background_gradient(cmap='RdBu_r', subset=['pct_diff_mean'])
)

# Gráfico de barras: top 10 features por diferencia absoluta
top10 = profile.head(10).copy()
fig, ax = plt.subplots(figsize=(10, 6))
colors_bar = ['#F44336' if x > 0 else '#2196F3' for x in top10['pct_diff_mean']]
ax.barh(range(len(top10)), top10['pct_diff_mean'], color=colors_bar, edgecolor='white')
ax.set_yticks(range(len(top10)))
ax.set_yticklabels(top10.index, fontsize=9)
ax.set_xlabel('% diferencia (decil 1 vs decil 10)')
ax.set_title('Top 10 features: diferencia decil 1 vs decil 10')
ax.axvline(0, color='black', linewidth=0.8)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# Resumen ejecutivo para el perfil
print(f'\n--- Resumen del perfil ---')
print(f'Features donde el decil 1 es MAYOR que el decil 10 (posibles señales de fraude):')
higher = top10[top10['pct_diff_mean'] > 0]
for _, row in higher.iterrows():
    print(f'  {row.name:40s}: +{row["pct_diff_mean"]:.0f}%')
print(f'\nFeatures donde el decil 1 es MENOR que el decil 10 (posibles señales de normalidad):')
lower = top10[top10['pct_diff_mean'] < 0]
for _, row in lower.iterrows():
    print(f'  {row.name:40s}: {row["pct_diff_mean"]:.0f}%')


## 8b. Zona gris — Clientes near-threshold (0.4–0.6)

Los clientes con probabilidad entre 0.4 y 0.6 están en la **zona de incertidumbre** del modelo: no son claramente fraudulentos ni claramente legítimos. Estos son los casos donde la revisión humana aporta más valor, porque el modelo no logra decidir.

**Qué analizamos:**
- ¿Cuántos clientes están en esta franja y qué porcentaje representan?
- ¿Se concentran en regiones específicas?
- ¿Su perfil de consumo es distinto al de los clientes de alta confianza (prob > 0.8)?
- ¿Qué features tienen valores intermedios (ni claramente fraudulentos ni claramente normales)?

**Implicancia operativa:** si hay muchos clientes en zona gris, el modelo solo está resolviendo los casos fáciles. Para los casos difíciles necesitás complementar con criterios de negocio, reglas expertas, o features adicionales.

In [ ]:
# --- Zona gris: clientes con probabilidad entre 0.4 y 0.6 ---
GRAY_LOW, GRAY_HIGH = 0.4, 0.6
gray_zone = df[(df['probability'] >= GRAY_LOW) & (df['probability'] <= GRAY_HIGH)]
high_conf = df[df['probability'] >= 0.8]

print(f'Clientes en zona gris ({GRAY_LOW}-{GRAY_HIGH}): {len(gray_zone):,} ({100*len(gray_zone)/len(df):.1f}%)')
print(f'Clientes alta confianza (>0.8):      {len(high_conf):,} ({100*len(high_conf)/len(df):.1f}%)')
print(f'Clientes baja confianza (<0.2):       {(df["probability"] < 0.2).sum():,} ({100*(df["probability"] < 0.2).sum()/len(df):.1f}%)')

# Distribución por región
print(f'\n--- Zona gris por región ---')
if 'geo_region' in df.columns:
    for region in sorted(df['geo_region'].dropna().unique()):
        region_total = (df['geo_region'] == region).sum()
        region_gray = ((df['geo_region'] == region) & (df['probability'] >= GRAY_LOW) & (df['probability'] <= GRAY_HIGH)).sum()
        print(f'  {region:20s}: {region_gray:5,} / {region_total:5,} ({100*region_gray/region_total:4.1f}%)')

# Boxplot: consumo por bucket de confianza
df['confidence_zone'] = pd.cut(
    df['probability'],
    bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0],
    labels=['<0.2 (conf. baja)', '0.2-0.4', '0.4-0.6 (gris)', '0.6-0.8', '>0.8 (conf. alta)']
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
zone_order = ['<0.2 (conf. baja)', '0.2-0.4', '0.4-0.6 (gris)', '0.6-0.8', '>0.8 (conf. alta)']
colors_zones = ['#2196F3', '#64B5F6', '#FFC107', '#FF9800', '#F44336']

# Count per zone
zone_counts = df['confidence_zone'].value_counts().reindex(zone_order)
axes[0].bar(range(len(zone_counts)), zone_counts.values, color=colors_zones, edgecolor='white')
axes[0].set_xticks(range(len(zone_counts)))
axes[0].set_xticklabels(zone_counts.index, rotation=30, ha='right', fontsize=9)
axes[0].set_title('Clientes por zona de confianza')
axes[0].set_ylabel('Cantidad de clientes')
for i, v in enumerate(zone_counts.values):
    axes[0].text(i, v + max(zone_counts)*0.01, f'{v:,}', ha='center', fontsize=9)

# Consumption last month by zone
consumo_data = [df[df['confidence_zone'] == z]['1_anterior'].dropna() for z in zone_order]
bp = axes[1].boxplot(consumo_data, tick_labels=zone_order, patch_artist=True, showfliers=False)
for patch, color in zip(bp['boxes'], colors_zones):
    patch.set_facecolor(color)
    patch.set_alpha(0.5)
axes[1].set_title('Consumo último mes por zona de confianza')
axes[1].set_ylabel('Consumo (kWh)')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

# Perfil de features: comparar zona gris vs alta confianza
print(f'\n--- Comparación de features: zona gris vs alta confianza ---')
feature_cols = [c for c in df.columns if c not in ['probability', 'cliente', 'confidence_zone', 'decile', 'prob_bucket', 'geo_region']]
numeric_features = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
# Limitar a 30 features más relevantes para no saturar
numeric_features = numeric_features[:30]

comparison = pd.DataFrame({
    'gray_mean': gray_zone[numeric_features].mean(),
    'high_mean': high_conf[numeric_features].mean(),
})
comparison['diff'] = comparison['high_mean'] - comparison['gray_mean']
comparison['pct_diff'] = ((comparison['high_mean'] - comparison['gray_mean']) / comparison['gray_mean'].abs().replace(0, np.nan)) * 100
comparison = comparison.sort_values('pct_diff', key=abs, ascending=False).dropna()

print(f'Top 10 features con mayor diferencia % (high_conf vs gray):')
display(comparison.head(10).style.format('{:.3f}').background_gradient(cmap='RdBu_r', subset=['pct_diff']))


## 9. Summary ejecutivo

Tablero consolidado con todas las métricas clave en un solo lugar. Ideal para copiar y pegar en un informe o slide.

**Incluye:**
- Totales de clientes evaluados, regiones, rango y media de probabilidad.
- Marcados por el threshold default y por thresholds alternativos (0.5, 0.7, 0.8, 0.9).
- Breakdown por región con clientes, media, marcados y alto riesgo.
- Timestamp y hash del modelo para trazabilidad.

In [ ]:
print("=" * 60)
print("SUMMARY — Inference Analysis")
print("=" * 60)
print(f"\nTotal clientes evaluados:    {len(df):,}")
print(f"Regiones:                     {', '.join(sorted(df['geo_region'].unique()))}")
print(f"Rango probabilidad:           [{prob.min():.6f}, {prob.max():.6f}]")
print(f"Media probabilidad:           {prob.mean():.4f}")
print(f"Mediana probabilidad:         {prob.median():.4f}")
print()
print(f"Threshold default ({default_threshold}):")
print(f"  Marcados:                   {marked:,} ({marked / len(prob) * 100:.2f}%)")
print(f"  No marcados:                {len(prob) - marked:,} ({(len(prob) - marked) / len(prob) * 100:.2f}%)")
print()

for t in [0.5, 0.7, 0.8, 0.9]:
    n = (prob >= t).sum()
    print(f"Prob >= {t}:                   {n:,} ({n / len(prob) * 100:.2f}%)")

print()
for region in sorted(df["geo_region"].unique()):
    rp = df[df["geo_region"] == region]["probability"]
    print(f"{region}:")
    print(f"  Clientes:                   {len(rp):,}")
    print(f"  Prob media:                 {rp.mean():.4f}")
    print(f"  Marcados (>=0.5):           {(rp >= 0.5).sum():,} ({(rp >= 0.5).mean() * 100:.2f}%)")
    print(f"  Alto riesgo (>=0.9):        {(rp >= 0.9).sum():,} ({(rp >= 0.9).mean() * 100:.2f}%)")

print(f"\nTimestamp inferencia:         {metadata['timestamp']}")
print(f"Model hash:                    {metadata['model_hash'][:16]}...")
print("=" * 60)

## 10. Export

Exporta los resultados del análisis a CSV para compartir con el equipo o cargar en otras herramientas (Excel, Power BI, etc.).

**Archivos generados:**
- `top_1000_suspicious.csv` — Los 1000 clientes más sospechosos con sus consumos.
- `decile_summary.csv` — Resumen estadístico por decil.
- `predictions_enriched.csv` — Todos los clientes con decil, bucket de probabilidad y meses en cero asignados.

In [ ]:
EXPORT_DIR = Path(INFERENCE_DIR) / "analysis"
EXPORT_DIR.mkdir(exist_ok=True)

# Top 1000 más sospechosos
top1k = df.nlargest(1000, "probability")[
    ["cliente", "geo_region", "probability", "decile"] + CONSUMPTION_COLS
]
path = EXPORT_DIR / "top_1000_suspicious.csv"
top1k.to_csv(path, index=False)
print(f"✅ {path}")

# Resumen por decil
path = EXPORT_DIR / "decile_summary.csv"
decile_summary.to_csv(path)
print(f"✅ {path}")

# Datos completos con decil y bucket asignados
path = EXPORT_DIR / "predictions_enriched.csv"
df[["cliente", "geo_region", "probability", "decile", "prob_bucket", "meses_cero"]].to_csv(path, index=False)
print(f"✅ {path}")

print(f"\nArchivos exportados a: {EXPORT_DIR.resolve()}")

---
## Notas

- Para usar con otra corrida de inferencia, solo cambiá las variables `INFERENCE_DIR` y `SEGMENT_THRESHOLDS_PATH` en la celda 0.
- Si no tenés segment thresholds, la sección 4 mostrará la comparación con el threshold global nomás.
- Los scatters usan un sample de 20k puntos por performance; ajustá `sample_n` si querés más.
- Los boxplots de consumo excluyen outliers (`showfliers=False`) para que las cajas sean legibles; los outliers extremos (ej. grandes industrias) pueden distorsionar la escala.